# SQL Practice: GROUP BY, Aggregates & HAVING (Extended)

This notebook expands the original optional HAVING practice into a full set of progressive exercises using the **Chinook** database.

**Focus areas:**
- Aggregation (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`)
- `GROUP BY`
- `HAVING` (filtering groups after aggregation)
- Simple and multi-table queries with `JOIN`

These exercises are designed for job-ready practice. Try each query yourself before looking at the solution.

## Data Schema

The schema below is the correct one for these exercises (slightly different from some demo environments).

![Database Schema](data-schema.png)

### Key Tables & Relationships (quick reference)

| Table | Important Columns | Notes |
|-------|-------------------|-------|
| `artists` | `ArtistId`, `Name` | |
| `albums` | `AlbumId`, `Title`, `ArtistId` | FK → artists |
| `tracks` | `TrackId`, `Name`, `AlbumId`, `GenreId`, `MediaTypeId`, `UnitPrice`, `Milliseconds` | FK → albums, genres, media_types |
| `genres` | `GenreId`, `Name` | |
| `media_types` | `MediaTypeId`, `Name` | |
| `playlists` | `PlaylistId`, `Name` | |
| `playlist_track` | `PlaylistId`, `TrackId` | Junction table |
| `customers` | `CustomerId`, `FirstName`, `LastName`, `Country`, `SupportRepId` | |
| `employees` | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` | |
| `invoices` | `InvoiceId`, `CustomerId`, `InvoiceDate`, `Total`, `BillingCountry` | |
| `invoice_items` | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` | |

---
## Exercise 1: Minimum and Maximum Track Prices per Album (Original)

Many albums have both their lowest and highest track prices equal to 99 cents.  
For marketing purposes, show only albums where the **maximum** track price is higher than $0.99.

### Instructions
Write a query that:
- Groups tracks by `AlbumId`
- Returns `AlbumId`, lowest track price (`MIN`), and highest track price (`MAX`) of `UnitPrice`
- Filters to albums where the highest track price is **above 0.99**
- Uses aliases `LowestTrackPrice` and `HighestTrackPrice`

In [ ]:
-- Your query here


**Expected output (partial):**
```
+---------+------------------+-------------------+
| AlbumId | LowestTrackPrice | HighestTrackPrice |
+---------+------------------+-------------------+
| 226     | 1.99             | 1.99              |
| 227     | 1.99             | 1.99              |
| ...     | ...              | ...               |
+---------+------------------+-------------------+
```

<details>
<summary><b>Hints</b></summary>

- Use `MIN(UnitPrice)` and `MAX(UnitPrice)`
- `GROUP BY AlbumId`
- Filter groups with `HAVING HighestTrackPrice > 0.99` (or the aggregate expression)
- Alias with `AS`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT AlbumId,
       MIN(UnitPrice) AS LowestTrackPrice,
       MAX(UnitPrice) AS HighestTrackPrice
FROM tracks
GROUP BY AlbumId
HAVING HighestTrackPrice > 0.99;
```

**Note:** In SQLite you can reference the alias in `HAVING`. In some other engines you must repeat the aggregate: `HAVING MAX(UnitPrice) > 0.99`.

</details>

---
## Exercise 2: Albums with Many Tracks

Find albums that contain **more than 15 tracks**.

### Instructions
- Group by `AlbumId`
- Return `AlbumId` and the number of tracks (`TrackCount`)
- Keep only albums with more than 15 tracks
- Order by `TrackCount` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `COUNT(*)` or `COUNT(TrackId)`
- `GROUP BY AlbumId`
- `HAVING TrackCount > 15` (or `HAVING COUNT(*) > 15`)
- `ORDER BY TrackCount DESC`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT AlbumId,
       COUNT(*) AS TrackCount
FROM tracks
GROUP BY AlbumId
HAVING TrackCount > 15
ORDER BY TrackCount DESC;
```

</details>

---
## Exercise 3: Genres with High Average Track Price

Which genres have an **average track price greater than $0.99**?

### Instructions
- Join `tracks` with `genres`
- Group by genre name
- Return genre name and the average unit price (alias `AvgPrice`)
- Keep only genres whose average price is above 0.99
- Order by `AvgPrice` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `JOIN genres ON tracks.GenreId = genres.GenreId`
- `AVG(UnitPrice)`
- `GROUP BY genres.Name` (or `g.Name` if you alias the table)
- `HAVING AvgPrice > 0.99`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT g.Name AS Genre,
       AVG(t.UnitPrice) AS AvgPrice
FROM tracks t
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
HAVING AvgPrice > 0.99
ORDER BY AvgPrice DESC;
```

</details>

---
## Exercise 4: Customers Who Spent More Than $40

Find customers whose **total invoice amount** exceeds $40.

### Instructions
- Join `customers` and `invoices`
- Group by customer
- Return CustomerId, full name (FirstName + ' ' + LastName), and total spent (`TotalSpent`)
- Keep only customers with `TotalSpent > 40`
- Order by `TotalSpent` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `SUM(i.Total)`
- Concatenate names: `c.FirstName || ' ' || c.LastName` (SQLite) or `CONCAT(...)` in other engines
- `GROUP BY c.CustomerId, c.FirstName, c.LastName` (or just the id if the engine allows)
- `HAVING TotalSpent > 40`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT c.CustomerId,
       c.FirstName || ' ' || c.LastName AS CustomerName,
       SUM(i.Total) AS TotalSpent
FROM customers c
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
HAVING TotalSpent > 40
ORDER BY TotalSpent DESC;
```

</details>

---
## Exercise 5: Artists with Multiple Albums and High Track Count

Find artists who have **at least 3 albums** and whose total number of tracks across all albums is **greater than 50**.

### Instructions
- Join `artists` → `albums` → `tracks`
- Group by artist
- Return Artist name, number of albums (`AlbumCount`), and total tracks (`TrackCount`)
- Keep only artists with `AlbumCount >= 3` **and** `TrackCount > 50`
- Order by `TrackCount` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `COUNT(DISTINCT a.AlbumId)` for album count
- `COUNT(t.TrackId)` for track count
- Two conditions in `HAVING`: `AlbumCount >= 3 AND TrackCount > 50`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT ar.Name AS Artist,
       COUNT(DISTINCT al.AlbumId) AS AlbumCount,
       COUNT(t.TrackId) AS TrackCount
FROM artists ar
JOIN albums al ON ar.ArtistId = al.ArtistId
JOIN tracks t ON al.AlbumId = t.AlbumId
GROUP BY ar.ArtistId, ar.Name
HAVING AlbumCount >= 3 AND TrackCount > 50
ORDER BY TrackCount DESC;
```

</details>

---
## Exercise 6: Countries with Significant Sales Volume

Which billing countries have **more than 10 invoices** and a **total sales amount greater than $100**?

### Instructions
- Use the `invoices` table
- Group by `BillingCountry`
- Return country, invoice count (`InvoiceCount`), and total sales (`TotalSales`)
- Apply both conditions with `HAVING`
- Order by `TotalSales` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `COUNT(InvoiceId)` and `SUM(Total)`
- `GROUP BY BillingCountry`
- `HAVING InvoiceCount > 10 AND TotalSales > 100`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT BillingCountry,
       COUNT(InvoiceId) AS InvoiceCount,
       SUM(Total) AS TotalSales
FROM invoices
GROUP BY BillingCountry
HAVING InvoiceCount > 10 AND TotalSales > 100
ORDER BY TotalSales DESC;
```

</details>

---
## Exercise 7: Playlists with Many Tracks (and Average Length)

Find playlists that contain **more than 20 tracks**. Also show the average track length in milliseconds.

### Instructions
- Join `playlists` → `playlist_track` → `tracks`
- Group by playlist
- Return playlist name, track count (`TrackCount`), and average milliseconds (`AvgMilliseconds`)
- Keep only playlists with more than 20 tracks
- Order by `TrackCount` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- Three-table join via the junction table `playlist_track`
- `COUNT(t.TrackId)` and `AVG(t.Milliseconds)`
- `HAVING TrackCount > 20`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT p.Name AS Playlist,
       COUNT(t.TrackId) AS TrackCount,
       AVG(t.Milliseconds) AS AvgMilliseconds
FROM playlists p
JOIN playlist_track pt ON p.PlaylistId = pt.PlaylistId
JOIN tracks t ON pt.TrackId = t.TrackId
GROUP BY p.PlaylistId, p.Name
HAVING TrackCount > 20
ORDER BY TrackCount DESC;
```

</details>

---
## Exercise 8: Support Representatives Performance

Which support representatives (employees) support customers who have generated **more than $100** in total sales?

### Instructions
- Join `employees` → `customers` → `invoices`
- Group by the support representative
- Return employee name, number of customers supported (`CustomerCount`), and total sales generated by their customers (`TotalSales`)
- Keep only reps whose customers generated more than $100 in total
- Order by `TotalSales` descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `employees.EmployeeId = customers.SupportRepId`
- `COUNT(DISTINCT c.CustomerId)` to avoid double-counting customers
- `SUM(i.Total)`
- `HAVING TotalSales > 100`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT e.FirstName || ' ' || e.LastName AS SupportRep,
       COUNT(DISTINCT c.CustomerId) AS CustomerCount,
       SUM(i.Total) AS TotalSales
FROM employees e
JOIN customers c ON e.EmployeeId = c.SupportRepId
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY e.EmployeeId, e.FirstName, e.LastName
HAVING TotalSales > 100
ORDER BY TotalSales DESC;
```

</details>

---
## Challenge Exercise 9: Top Genres by Revenue (with HAVING)

Calculate revenue per genre (using `invoice_items` + `tracks` + `genres`).  
Show only genres that generated **more than $50** in revenue.

### Instructions
- Join `invoice_items` → `tracks` → `genres`
- Revenue = `SUM(ii.UnitPrice * ii.Quantity)`
- Group by genre name
- Return genre and total revenue (`Revenue`)
- Keep genres with `Revenue > 50`
- Order by revenue descending

In [ ]:
-- Your query here


<details>
<summary><b>Hints</b></summary>

- `SUM(ii.UnitPrice * ii.Quantity)`
- Three-table join
- `HAVING Revenue > 50`

</details>

<details>
<summary><b>Solution</b></summary>

```sql
SELECT g.Name AS Genre,
       SUM(ii.UnitPrice * ii.Quantity) AS Revenue
FROM invoice_items ii
JOIN tracks t ON ii.TrackId = t.TrackId
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
HAVING Revenue > 50
ORDER BY Revenue DESC;
```

</details>

---
## Challenge Exercise 10: Albums Whose Tracks Are All Expensive

Find albums where **every track** costs more than $0.99 (i.e., the minimum track price in the album is greater than 0.99).

### Instructions
- Group by album
- Return `AlbumId` and `MIN(UnitPrice)` as `MinPrice`
- Keep only albums where the cheapest track is still > 0.99
- This is a classic use of `HAVING` with `MIN`

In [ ]:
-- Your query here


<details>
<summary><b>Solution</b></summary>

```sql
SELECT AlbumId,
       MIN(UnitPrice) AS MinPrice
FROM tracks
GROUP BY AlbumId
HAVING MinPrice > 0.99;
```

This is the logical counterpart of Exercise 1.

</details>

---
## Quick Reference: WHERE vs HAVING

| Clause   | Filters on          | When it runs          | Can use aggregates? |
|----------|---------------------|-----------------------|---------------------|
| `WHERE`  | Individual rows     | Before grouping       | No                  |
| `HAVING` | Groups / aggregates | After `GROUP BY`      | Yes                 |

**Rule of thumb:**
- Filter **rows** → `WHERE`
- Filter **groups** (after aggregation) → `HAVING`

You can (and often should) use both in the same query.

---
## Extra Practice Ideas

Once you finish the exercises above, try these variations on your own:

1. Genres that have more than 100 tracks **and** average length > 200000 ms.
2. Customers from the USA who spent more than $20.
3. Artists who have albums in more than one genre.
4. Media types used by more than 500 tracks.
5. Employees who manage other employees (using the self-relationship on `ReportsTo`) and also support customers with total sales > a threshold.

Happy querying! 🚀